# Analysis of CERF Humanitarian Funding Allocations in Nigeria

**Data Analytics Capstone Project**

**Author:** Kimto Oche Emmanuel

This project examines the distribution of Central Emergency Response Fund allocations in Nigeria across agencies, emergencies, sectors, funding windows and years.

In [1]:
print("Python kernel is working.")

Python kernel is working.


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# Identify the project folder and raw dataset
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

raw_data_path = (
    project_root
    / "data"
    / "raw"
    / "Nigeria_CERF-Allocation.csv"
)

print("Dataset path:", raw_data_path)
print("Dataset exists:", raw_data_path.exists())

# Load the original CSV into a Pandas DataFrame
df_raw = pd.read_csv(raw_data_path)

print("Dataset loaded successfully.")
print("Number of rows:", df_raw.shape[0])
print("Number of columns:", df_raw.shape[1])

Dataset path: /Users/test/Documents/cerf-nigeria-capstone/data/raw/Nigeria_CERF-Allocation.csv
Dataset exists: True
Dataset loaded successfully.
Number of rows: 125
Number of columns: 17


## 1. Data Loading and Initial Inspection

The original CSV dataset was loaded without modification. Initial inspection was conducted to understand its structure, variables, data types and completeness before cleaning.

In [4]:
# Display the first five records
df_raw.head()

,projectID,agencyName,continentName,countryCode,countryName,dateUSGSignature,emergencyTypeName,projectCode,projectTitle,regionName,totalAmountApproved,windowFullName,year,projectsectors,projectclusters,projectgroupings,projectcapcodes
0,1770,World Health Organization,Africa,NGA,Nigeria,5/1/2009,Unspecified Health Emergency,09-WHO-028,Project for emergency health intervention to c...,Western Africa,1279887.0,Rapid Response,2009,Health,Health,NaN,NaN
1,2408,United Nations Children’s Fund,Africa,NGA,Nigeria,8/27/2010,Multiple Emergencies,10-CEF-044,Response to an Outbreak of lead Poisoning in Z...,Western Africa,1181590.0,Rapid Response,2010,Health,Health,NaN,NaN
2,2409,World Health Organization,Africa,NGA,Nigeria,9/2/2010,Multiple Emergencies,10-WHO-052,Response to an Outbreak of lead Poisoning in Z...,Western Africa,817612.0,Rapid Response,2010,Health,Health,NaN,NaN
3,3738,United Nations Children’s Fund,Africa,NGA,Nigeria,1/17/2013,Flood,13-CEF-001,Life-saving WASH Interventions for Flood Affec...,Western Africa,1867213.0,Rapid Response,2013,"Water, Sanitation and Hygiene","Water, Sanitation and Hygiene",NaN,NaN
4,3739,United Nations Children’s Fund,Africa,NGA,Nigeria,1/21/2013,Flood,13-CEF-002,"Emergency health care, prevention and response...",Western Africa,1099462.0,Rapid Response,2013,Health,Health,NaN,NaN


In [5]:
# Summarise the structure and completeness of every column
structure_summary = pd.DataFrame({
    "column_name": df_raw.columns,
    "data_type": df_raw.dtypes.astype(str).values,
    "non_null_values": df_raw.notna().sum().values,
    "missing_values": df_raw.isna().sum().values,
    "unique_values": df_raw.nunique(dropna=True).values
})

structure_summary

,column_name,data_type,non_null_values,missing_values,unique_values
0,projectID,int64,125,0,125
1,agencyName,str,125,0,8
2,continentName,str,125,0,1
3,countryCode,str,125,0,1
4,countryName,str,125,0,1
5,dateUSGSignature,str,125,0,75
6,emergencyTypeName,str,125,0,9
7,projectCode,str,125,0,125
8,projectTitle,str,125,0,121
9,regionName,str,125,0,1


## 2. Initial Data-Quality Assessment

The dataset was assessed for missing values, exact duplicate rows, repeated identifiers, date-conversion problems and year inconsistencies before any cleaning changes were made.

In [6]:
# Calculate missing-value counts and percentages
missing_summary = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_percentage": (df_raw.isna().mean() * 100).round(1)
})

# Display only columns containing missing values
missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
]

missing_summary

,missing_count,missing_percentage
projectgroupings,64,51.2
projectcapcodes,87,69.6


In [7]:
print("Exact duplicate rows:", df_raw.duplicated().sum())
print(
    "Duplicate project IDs:",
    df_raw["projectID"].duplicated().sum()
)
print(
    "Duplicate project codes:",
    df_raw["projectCode"].duplicated().sum()
)
print(
    "Repeated project-title entries:",
    df_raw["projectTitle"].duplicated().sum()
)

Exact duplicate rows: 0
Duplicate project IDs: 0
Duplicate project codes: 0
Repeated project-title entries: 4


In [8]:
# Temporarily convert the date field for quality checking
parsed_signature_dates = pd.to_datetime(
    df_raw["dateUSGSignature"],
    errors="coerce"
)

print(
    "Dates that could not be converted:",
    parsed_signature_dates.isna().sum()
)
print(
    "Earliest signature date:",
    parsed_signature_dates.min()
)
print(
    "Latest signature date:",
    parsed_signature_dates.max()
)
print(
    "Year mismatches:",
    (
        df_raw["year"]
        != parsed_signature_dates.dt.year
    ).sum()
)
print(
    "Years represented:",
    sorted(df_raw["year"].unique())
)

Dates that could not be converted: 0
Earliest signature date: 2009-05-01 00:00:00
Latest signature date: 2026-06-03 00:00:00
Year mismatches: 0
Years represented: [np.int64(2009), np.int64(2010), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


In [9]:
funding = df_raw["totalAmountApproved"]

print(f"Number of projects: {funding.count():,}")
print(f"Total approved funding: ${funding.sum():,.2f}")
print(f"Average project funding: ${funding.mean():,.2f}")
print(f"Median project funding: ${funding.median():,.2f}")
print(f"Minimum project funding: ${funding.min():,.2f}")
print(f"Maximum project funding: ${funding.max():,.2f}")
print(f"Standard deviation: ${funding.std():,.2f}")

Number of projects: 125
Total approved funding: $207,838,943.82
Average project funding: $1,662,711.55
Median project funding: $1,059,082.00
Minimum project funding: $197,526.00
Maximum project funding: $15,000,005.00
Standard deviation: $2,000,380.28


In [10]:
category_columns = [
    "agencyName",
    "emergencyTypeName",
    "windowFullName"
]

for column in category_columns:
    print(f"\n{column}")
    print(df_raw[column].value_counts().to_string())


agencyName
agencyName
United Nations Children’s Fund                   39
World Health Organization                        17
World Food Programme                             15
United Nations Population Fund                   14
International Organization for Migration         14
United Nations High Commissioner for Refugees    13
Food and Agriculture Organization                10
United Nations Development Programme              3

emergencyTypeName
emergencyTypeName
Displacement                    73
Flood                           23
Violence/Clashes                 8
Cholera                          7
Economic Disruption              4
Unspecified Health Emergency     3
Drought                          3
Multiple Emergencies             2
Ebola                            2

windowFullName
windowFullName
Rapid Response             88
Underfunded Emergencies    37


## 3. Data Cleaning and Feature Engineering

A separate copy of the raw dataset was created for cleaning. This preserved the original data while allowing column standardisation, date conversion, missing-value treatment and the creation of analytical variables.

In [11]:
# Preserve the original DataFrame by creating a separate cleaning copy
df_clean = df_raw.copy(deep=True)

# Standardise column names using clear snake_case names
column_name_mapping = {
    "projectID": "project_id",
    "agencyName": "agency_name",
    "continentName": "continent_name",
    "countryCode": "country_code",
    "countryName": "country_name",
    "dateUSGSignature": "date_usg_signature",
    "emergencyTypeName": "emergency_type_name",
    "projectCode": "project_code",
    "projectTitle": "project_title",
    "regionName": "region_name",
    "totalAmountApproved": "total_amount_approved",
    "windowFullName": "window_full_name",
    "year": "year",
    "projectsectors": "project_sectors",
    "projectclusters": "project_clusters",
    "projectgroupings": "project_groupings",
    "projectcapcodes": "project_cap_codes"
}

df_clean = df_clean.rename(columns=column_name_mapping)

# Check for and remove exact duplicate rows
rows_before_cleaning = len(df_clean)
df_clean = df_clean.drop_duplicates().copy()
exact_duplicates_removed = rows_before_cleaning - len(df_clean)

# Remove unnecessary leading, trailing and repeated whitespace
text_columns = [
    "agency_name",
    "continent_name",
    "country_code",
    "country_name",
    "emergency_type_name",
    "project_code",
    "project_title",
    "region_name",
    "window_full_name",
    "project_sectors",
    "project_clusters",
    "project_groupings",
    "project_cap_codes"
]

for column in text_columns:
    df_clean[column] = (
        df_clean[column]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# Convert the signature date from text to datetime
df_clean["date_usg_signature"] = pd.to_datetime(
    df_clean["date_usg_signature"],
    errors="coerce"
)

# Create useful date variables
df_clean["signature_year"] = (
    df_clean["date_usg_signature"].dt.year
)
df_clean["signature_month"] = (
    df_clean["date_usg_signature"].dt.month
)
df_clean["signature_month_name"] = (
    df_clean["date_usg_signature"].dt.month_name()
)

# Flag disagreement between the supplied year and signature year
df_clean["year_mismatch_flag"] = (
    df_clean["year"] != df_clean["signature_year"]
)

# Replace only genuinely missing grouping and CAP-code values
df_clean["project_groupings"] = (
    df_clean["project_groupings"]
    .replace("", pd.NA)
    .fillna("Not Specified")
)

df_clean["project_cap_codes"] = (
    df_clean["project_cap_codes"]
    .replace("", pd.NA)
    .fillna("Not Specified")
)

# Confirm that funding is numerical
df_clean["total_amount_approved"] = pd.to_numeric(
    df_clean["total_amount_approved"],
    errors="coerce"
)

# Create descriptive funding bands
# These are analytical categories, not official CERF classifications
funding_bins = [
    0,
    500_000,
    1_000_000,
    3_000_000,
    np.inf
]

funding_labels = [
    "Small (up to $500K)",
    "Medium ($500K to $1M)",
    "Large ($1M to $3M)",
    "Very Large (over $3M)"
]

df_clean["funding_category"] = pd.cut(
    df_clean["total_amount_approved"],
    bins=funding_bins,
    labels=funding_labels,
    include_lowest=True
)

print("Cleaning operations completed.")
print("Exact duplicate rows removed:", exact_duplicates_removed)
print("Raw dataset shape:", df_raw.shape)
print("Cleaned dataset shape:", df_clean.shape)

Cleaning operations completed.
Exact duplicate rows removed: 0
Raw dataset shape: (125, 17)
Cleaned dataset shape: (125, 22)


### Cleaning Validation

Validation checks were performed to confirm that cleaning preserved the records and funding totals, maintained unique identifiers, converted all dates successfully and resolved the documented missing fields.

In [12]:
# Calculate totals for reconciliation
raw_funding_total = df_raw[
    "totalAmountApproved"
].sum()

clean_funding_total = df_clean[
    "total_amount_approved"
].sum()

# Create validation tests
validation_results = {
    "Row count preserved": len(df_clean) == len(df_raw),
    "Project IDs remain unique": df_clean["project_id"].is_unique,
    "Project codes remain unique": df_clean["project_code"].is_unique,
    "No missing funding values": (
        df_clean["total_amount_approved"].isna().sum() == 0
    ),
    "All dates converted successfully": (
        df_clean["date_usg_signature"].isna().sum() == 0
    ),
    "No year mismatches": (
        df_clean["year_mismatch_flag"].sum() == 0
    ),
    "Grouping missing values resolved": (
        df_clean["project_groupings"].isna().sum() == 0
    ),
    "CAP-code missing values resolved": (
        df_clean["project_cap_codes"].isna().sum() == 0
    ),
    "Funding categories complete": (
        df_clean["funding_category"].isna().sum() == 0
    ),
    "Funding total reconciles": np.isclose(
        raw_funding_total,
        clean_funding_total,
        atol=0.01
    )
}

validation_table = pd.DataFrame(
    validation_results.items(),
    columns=["validation_check", "passed"]
)

display(validation_table)

print(
    "All validation checks passed:",
    validation_table["passed"].all()
)
print(
    "Raw funding total:",
    f"${raw_funding_total:,.2f}"
)
print(
    "Cleaned funding total:",
    f"${clean_funding_total:,.2f}"
)
print(
    "Not Specified groupings:",
    (
        df_clean["project_groupings"]
        == "Not Specified"
    ).sum()
)
print(
    "Not Specified CAP codes:",
    (
        df_clean["project_cap_codes"]
        == "Not Specified"
    ).sum()
)

,validation_check,passed
0,Row count preserved,True
1,Project IDs remain unique,True
2,Project codes remain unique,True
3,No missing funding values,True
4,All dates converted successfully,True
5,No year mismatches,True
6,Grouping missing values resolved,True
7,CAP-code missing values resolved,True
8,Funding categories complete,True
9,Funding total reconciles,True


All validation checks passed: True
Raw funding total: $207,838,943.82
Cleaned funding total: $207,838,943.82
Not Specified groupings: 64
Not Specified CAP codes: 87


In [13]:
# Create the processed-data file path
processed_data_path = (
    project_root
    / "data"
    / "processed"
    / "cerf_nigeria_cleaned.csv"
)

# Export without creating an unnecessary index column
df_clean.to_csv(
    processed_data_path,
    index=False,
    date_format="%Y-%m-%d"
)

print("Cleaned dataset exported successfully.")
print("Export path:", processed_data_path)
print("Exported file exists:", processed_data_path.exists())

Cleaned dataset exported successfully.
Export path: /Users/test/Documents/cerf-nigeria-capstone/data/processed/cerf_nigeria_cleaned.csv
Exported file exists: True


## 4. Exploratory Data Analysis

The cleaned dataset was aggregated to examine how approved funding and project activity were distributed across agencies, emergencies, sectors, funding windows and years.

In [14]:
# Aggregate funding statistics by implementing agency
agency_summary = (
    df_clean
    .groupby("agency_name", as_index=False)
    .agg(
        total_funding=(
            "total_amount_approved",
            "sum"
        ),
        project_count=(
            "project_id",
            "count"
        ),
        average_funding=(
            "total_amount_approved",
            "mean"
        ),
        median_funding=(
            "total_amount_approved",
            "median"
        )
    )
    .sort_values(
        "total_funding",
        ascending=False
    )
)

# Calculate each agency's percentage share of total funding
agency_summary["funding_share_percent"] = (
    agency_summary["total_funding"]
    / agency_summary["total_funding"].sum()
    * 100
)

# Display the formatted result
display(
    agency_summary.style.format({
        "total_funding": "${:,.2f}",
        "average_funding": "${:,.2f}",
        "median_funding": "${:,.2f}",
        "funding_share_percent": "{:.1f}%"
    })
)

,agency_name,total_funding,project_count,average_funding,median_funding,funding_share_percent
2,United Nations Children’s Fund,"$73,198,383.67",39,"$1,876,881.63","$1,409,088.00",35.2%
6,World Food Programme,"$60,787,655.87",15,"$4,052,510.39","$2,265,652.38",29.2%
1,International Organization for Migration,"$18,868,034.49",14,"$1,347,716.75","$1,136,591.27",9.1%
4,United Nations High Commissioner for Refugees,"$16,151,289.46",13,"$1,242,406.88","$1,109,375.00",7.8%
7,World Health Organization,"$15,110,931.16",17,"$888,878.30","$750,000.00",7.3%
0,Food and Agriculture Organization,"$12,962,920.28",10,"$1,296,292.03","$1,200,011.00",6.2%
5,United Nations Population Fund,"$10,004,963.89",14,"$714,640.28","$446,108.95",4.8%
3,United Nations Development Programme,"$754,765.00",3,"$251,588.33","$272,409.00",0.4%


In [15]:
agency_summary.to_csv(
    project_root
    / "outputs"
    / "tables"
    / "agency_funding_summary.csv",
    index=False
)

print("Agency summary saved successfully.")

Agency summary saved successfully.


In [16]:
# Create readable agency abbreviations for the chart
agency_abbreviations = {
    "United Nations Children’s Fund": "UNICEF",
    "World Health Organization": "WHO",
    "World Food Programme": "WFP",
    "United Nations Population Fund": "UNFPA",
    "International Organization for Migration": "IOM",
    "United Nations High Commissioner for Refugees": "UNHCR",
    "Food and Agriculture Organization": "FAO",
    "United Nations Development Programme": "UNDP"
}

agency_chart_data = agency_summary.sort_values(
    "total_funding",
    ascending=True
).copy()

agency_chart_data["agency_short"] = (
    agency_chart_data["agency_name"]
    .map(agency_abbreviations)
)

agency_chart_data["funding_label"] = (
    agency_chart_data["total_funding"]
    .map(lambda value: f"${value / 1_000_000:.1f}M")
)

fig_agency = px.bar(
    agency_chart_data,
    x="total_funding",
    y="agency_short",
    orientation="h",
    text="funding_label",
    title="CERF Funding by Implementing Agency",
    labels={
        "total_funding": "Total approved funding (US$)",
        "agency_short": ""
    },
    hover_data={
        "agency_name": True,
        "project_count": True,
        "average_funding": ":$,.2f",
        "funding_share_percent": ":.1f",
        "funding_label": False
    },
    color_discrete_sequence=["#185A8D"]
)

fig_agency.update_traces(
    textposition="outside",
    cliponaxis=False
)

fig_agency.update_layout(
    height=520,
    showlegend=False,
    margin=dict(l=70, r=80, t=80, b=70),
    xaxis=dict(
        tickprefix="$",
        tickformat=".2s",
        tickangle=0,
        automargin=True
    ),
    yaxis=dict(
        title="",
        automargin=True
    )
)

fig_agency.update_xaxes(
    range=[
        0,
        agency_chart_data["total_funding"].max() * 1.18
    ]
)

fig_agency.show()

In [17]:
# Measure funding concentration among the two largest agencies
top_two_agency_share = (
    agency_summary.head(2)["total_funding"].sum()
    / clean_funding_total
    * 100
)

print(
    "Top-two agency funding share:",
    f"{top_two_agency_share:.1f}%"
)

# Funding by emergency type
emergency_summary = (
    df_clean
    .groupby("emergency_type_name", as_index=False)
    .agg(
        total_funding=(
            "total_amount_approved",
            "sum"
        ),
        project_count=(
            "project_id",
            "count"
        ),
        average_funding=(
            "total_amount_approved",
            "mean"
        )
    )
    .sort_values(
        "total_funding",
        ascending=False
    )
)

emergency_summary["funding_share_percent"] = (
    emergency_summary["total_funding"]
    / clean_funding_total
    * 100
)

# Funding by CERF window
window_summary = (
    df_clean
    .groupby("window_full_name", as_index=False)
    .agg(
        total_funding=(
            "total_amount_approved",
            "sum"
        ),
        project_count=(
            "project_id",
            "count"
        ),
        average_funding=(
            "total_amount_approved",
            "mean"
        )
    )
    .sort_values(
        "total_funding",
        ascending=False
    )
)

window_summary["funding_share_percent"] = (
    window_summary["total_funding"]
    / clean_funding_total
    * 100
)

print("\nFunding by emergency type")
display(
    emergency_summary.style.format({
        "total_funding": "${:,.2f}",
        "average_funding": "${:,.2f}",
        "funding_share_percent": "{:.1f}%"
    })
)

print("\nFunding by CERF window")
display(
    window_summary.style.format({
        "total_funding": "${:,.2f}",
        "average_funding": "${:,.2f}",
        "funding_share_percent": "{:.1f}%"
    })
)

Top-two agency funding share: 64.5%

Funding by emergency type


,emergency_type_name,total_funding,project_count,average_funding,funding_share_percent
1,Displacement,"$109,403,629.00",73,"$1,498,679.85",52.6%
8,Violence/Clashes,"$37,500,000.70",8,"$4,687,500.09",18.0%
5,Flood,"$25,885,710.11",23,"$1,125,465.66",12.5%
0,Cholera,"$12,257,408.01",7,"$1,751,058.29",5.9%
2,Drought,"$11,000,000.48",3,"$3,666,666.83",5.3%
4,Economic Disruption,"$6,000,007.52",4,"$1,500,001.88",2.9%
7,Unspecified Health Emergency,"$2,334,677.00",3,"$778,225.67",1.1%
6,Multiple Emergencies,"$1,999,202.00",2,"$999,601.00",1.0%
3,Ebola,"$1,458,309.00",2,"$729,154.50",0.7%



Funding by CERF window


,window_full_name,total_funding,project_count,average_funding,funding_share_percent
0,Rapid Response,"$146,797,893.82",88,"$1,668,157.88",70.6%
1,Underfunded Emergencies,"$61,041,050.00",37,"$1,649,758.11",29.4%


In [18]:
emergency_summary.to_csv(
    project_root
    / "outputs"
    / "tables"
    / "emergency_funding_summary.csv",
    index=False
)

window_summary.to_csv(
    project_root
    / "outputs"
    / "tables"
    / "funding_window_summary.csv",
    index=False
)

print("Emergency and funding-window tables saved.")

Emergency and funding-window tables saved.


In [19]:
emergency_chart_data = emergency_summary.sort_values(
    "total_funding",
    ascending=True
).copy()

emergency_chart_data["funding_label"] = (
    emergency_chart_data["total_funding"]
    .map(lambda value: f"${value / 1_000_000:.1f}M")
)

fig_emergency = px.bar(
    emergency_chart_data,
    x="total_funding",
    y="emergency_type_name",
    orientation="h",
    text="funding_label",
    title="CERF Funding by Emergency Type",
    labels={
        "total_funding": "Total approved funding (US$)",
        "emergency_type_name": ""
    },
    hover_data={
        "project_count": True,
        "average_funding": ":$,.2f",
        "funding_share_percent": ":.1f",
        "funding_label": False
    },
    color_discrete_sequence=["#287D8E"]
)

fig_emergency.update_traces(
    textposition="outside",
    cliponaxis=False
)

fig_emergency.update_layout(
    height=560,
    showlegend=False,
    margin=dict(l=80, r=80, t=80, b=60),
    xaxis=dict(
        tickprefix="$",
        tickformat=".2s",
        tickangle=0
    ),
    yaxis=dict(title="", automargin=True)
)

fig_emergency.update_xaxes(
    range=[
        0,
        emergency_chart_data["total_funding"].max() * 1.20
    ]
)

fig_emergency.show()

In [20]:
fig_window = px.pie(
    window_summary,
    values="total_funding",
    names="window_full_name",
    hole=0.55,
    title="Funding Distribution by CERF Window",
    color_discrete_sequence=[
        "#185A8D",
        "#E9A23B"
    ],
    hover_data={
        "project_count": True,
        "average_funding": ":$,.2f"
    }
)

fig_window.update_traces(
    textposition="inside",
    textinfo="label+percent"
)

fig_window.update_layout(
    height=500,
    legend_title_text="Funding window"
)

fig_window.show()

In [21]:
# Aggregate funding and project activity by recorded year
year_summary = (
    df_clean
    .groupby("year", as_index=False)
    .agg(
        total_funding=(
            "total_amount_approved",
            "sum"
        ),
        project_count=(
            "project_id",
            "count"
        ),
        average_funding=(
            "total_amount_approved",
            "mean"
        )
    )
    .sort_values("year")
)

display(
    year_summary.style.format({
        "total_funding": "${:,.2f}",
        "average_funding": "${:,.2f}"
    })
)

year_summary.to_csv(
    project_root
    / "outputs"
    / "tables"
    / "annual_funding_summary.csv",
    index=False
)

print("Annual funding summary saved.")

,year,total_funding,project_count,average_funding
0,2009,"$1,279,887.00",1,"$1,279,887.00"
1,2010,"$1,999,202.00",2,"$999,601.00"
2,2013,"$6,431,433.00",6,"$1,071,905.50"
3,2014,"$5,004,954.00",10,"$500,495.40"
4,2015,"$9,889,075.00",8,"$1,236,134.38"
5,2016,"$23,483,769.00",21,"$1,118,274.71"
6,2017,"$31,886,628.00",22,"$1,449,392.18"
7,2018,"$6,866,877.00",8,"$858,359.62"
8,2020,"$13,001,946.00",4,"$3,250,486.50"
9,2021,"$33,500,110.00",12,"$2,791,675.83"


Annual funding summary saved.


In [22]:
# Create a complete year sequence
all_years = pd.DataFrame({
    "year": range(
        int(df_clean["year"].min()),
        int(df_clean["year"].max()) + 1
    )
})

# Missing values mean the supplied dataset contains no records
# for those years; they are not automatically treated as zero.
year_chart_data = all_years.merge(
    year_summary,
    on="year",
    how="left"
)

year_chart_data

,year,total_funding,project_count,average_funding
0,2009,1279887.00,1.0,1.279887e+06
1,2010,1999202.00,2.0,9.996010e+05
2,2011,NaN,NaN,NaN
3,2012,NaN,NaN,NaN
4,2013,6431433.00,6.0,1.071906e+06
5,2014,5004954.00,10.0,5.004954e+05
6,2015,9889075.00,8.0,1.236134e+06
7,2016,23483769.00,21.0,1.118275e+06
8,2017,31886628.00,22.0,1.449392e+06
9,2018,6866877.00,8.0,8.583596e+05


In [23]:
fig_year = px.line(
    year_chart_data,
    x="year",
    y="total_funding",
    markers=True,
    title="Annual CERF Funding Represented in the Dataset",
    labels={
        "year": "Signature year",
        "total_funding": "Total approved funding (US$)"
    },
    hover_data={
        "project_count": True,
        "average_funding": ":$,.2f"
    }
)

fig_year.update_traces(
    line=dict(
        color="#185A8D",
        width=3
    ),
    marker=dict(
        size=9,
        color="#E9A23B"
    ),
    connectgaps=False
)

partial_2026_funding = year_summary.loc[
    year_summary["year"] == 2026,
    "total_funding"
].iloc[0]

fig_year.add_annotation(
    x=2026,
    y=partial_2026_funding,
    text="2026 is a partial year",
    showarrow=True,
    arrowhead=2,
    ax=-90,
    ay=-50
)

fig_year.update_layout(
    height=520,
    showlegend=False,
    margin=dict(l=70, r=70, t=80, b=80),
    xaxis=dict(
        dtick=1,
        tickangle=-45
    ),
    yaxis=dict(
        tickprefix="$",
        tickformat=".2s"
    )
)

fig_year.show()

In [24]:
# Aggregate annual funding by CERF window
window_year_summary = (
    df_clean
    .groupby(
        ["year", "window_full_name"],
        as_index=False
    )
    .agg(
        total_funding=(
            "total_amount_approved",
            "sum"
        ),
        project_count=(
            "project_id",
            "count"
        )
    )
    .sort_values(
        ["year", "window_full_name"]
    )
)

display(
    window_year_summary.style.format({
        "total_funding": "${:,.2f}"
    })
)

window_year_summary.to_csv(
    project_root
    / "outputs"
    / "tables"
    / "annual_funding_by_window.csv",
    index=False
)

print("Annual funding-window summary saved.")

,year,window_full_name,total_funding,project_count
0,2009,Rapid Response,"$1,279,887.00",1
1,2010,Rapid Response,"$1,999,202.00",2
2,2013,Rapid Response,"$6,431,433.00",6
3,2014,Rapid Response,"$1,458,309.00",2
4,2014,Underfunded Emergencies,"$3,546,645.00",8
5,2015,Rapid Response,"$9,889,075.00",8
6,2016,Rapid Response,"$23,483,769.00",21
7,2017,Rapid Response,"$9,889,471.00",9
8,2017,Underfunded Emergencies,"$21,997,157.00",13
9,2018,Rapid Response,"$6,866,877.00",8


Annual funding-window summary saved.


In [25]:
window_colours = {
    "Rapid Response": "#185A8D",
    "Underfunded Emergencies": "#E9A23B"
}

fig_window_year = px.bar(
    window_year_summary,
    x="year",
    y="total_funding",
    color="window_full_name",
    barmode="stack",
    title="Annual Funding by CERF Window",
    labels={
        "year": "Signature year",
        "total_funding": "Total approved funding (US$)",
        "window_full_name": "Funding window"
    },
    color_discrete_map=window_colours,
    hover_data={
        "project_count": True,
        "total_funding": ":$,.2f"
    }
)

fig_window_year.update_layout(
    height=520,
    margin=dict(l=70, r=50, t=80, b=80),
    xaxis=dict(
        dtick=1,
        tickangle=-45
    ),
    yaxis=dict(
        tickprefix="$",
        tickformat=".2s"
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

fig_window_year.add_annotation(
    x=2026,
    y=partial_2026_funding,
    text="Partial year",
    showarrow=True,
    arrowhead=2,
    ax=-55,
    ay=-45
)



In [26]:
# Improve the title, legend and chart spacing
fig_window_year.update_layout(
    height=580,
    title=dict(
        text="Annual Funding by CERF Window",
        x=0.5,
        xanchor="center",
        y=0.97,
        yanchor="top"
    ),
    margin=dict(
        l=70,
        r=50,
        t=110,
        b=150
    ),
    legend=dict(
        orientation="h",
        title_text="",
        yanchor="top",
        y=-0.22,
        xanchor="center",
        x=0.5
    ),
    xaxis=dict(
        dtick=1,
        tickangle=-45,
        title="Signature year"
    ),
    yaxis=dict(
        tickprefix="$",
        tickformat=".2s",
        title="Total approved funding (US$)"
    )
)

fig_window_year.show()

### Sector Methodology

Some records contain multiple sectors in a single project-sector field. To prevent double-counting, funding was aggregated using the original sector combinations exactly as recorded. The full project amount was not assigned repeatedly to every individual sector.

In [27]:
sector_summary = (
    df_clean
    .groupby("project_sectors", as_index=False)
    .agg(
        total_funding=(
            "total_amount_approved",
            "sum"
        ),
        project_count=(
            "project_id",
            "count"
        ),
        average_funding=(
            "total_amount_approved",
            "mean"
        ),
        years_with_funding=(
            "year",
            "nunique"
        )
    )
    .sort_values(
        "total_funding",
        ascending=False
    )
)

sector_summary["funding_share_percent"] = (
    sector_summary["total_funding"]
    / clean_funding_total
    * 100
)

print("Top ten sector combinations by total funding")

display(
    sector_summary.head(10).style.format({
        "total_funding": "${:,.2f}",
        "average_funding": "${:,.2f}",
        "funding_share_percent": "{:.1f}%"
    })
)

Top ten sector combinations by total funding


,project_sectors,total_funding,project_count,average_funding,years_with_funding,funding_share_percent
11,Food Assistance,"$46,820,063.35",6,"$7,803,343.89",6,22.5%
12,Health,"$22,923,953.16",30,"$764,131.77",10,11.0%
14,"Health, Nutrition, Water, Sanitation and Hygiene","$19,900,007.11",4,"$4,975,001.78",3,9.6%
19,Nutrition,"$18,098,688.95",9,"$2,010,965.44",6,8.7%
23,"Water, Sanitation and Hygiene","$14,231,381.75",8,"$1,778,922.72",6,6.8%
7,Common Services,"$12,956,704.70",11,"$1,177,882.25",6,6.2%
22,Shelter and Non-Food Items,"$12,878,151.95",9,"$1,430,905.77",7,6.2%
20,Protection,"$12,343,400.46",21,"$587,780.97",7,5.9%
0,Agriculture,"$11,572,294.31",8,"$1,446,536.79",8,5.6%
17,"Health, Water, Sanitation and Hygiene","$6,100,000.01",2,"$3,050,000.00",2,2.9%


In [28]:
crisis_summary = (
    df_clean
    .groupby("project_groupings", as_index=False)
    .agg(
        total_funding=(
            "total_amount_approved",
            "sum"
        ),
        project_count=(
            "project_id",
            "count"
        ),
        average_funding=(
            "total_amount_approved",
            "mean"
        )
    )
    .sort_values(
        "total_funding",
        ascending=False
    )
)

crisis_summary["funding_share_percent"] = (
    crisis_summary["total_funding"]
    / clean_funding_total
    * 100
)

display(
    crisis_summary.style.format({
        "total_funding": "${:,.2f}",
        "average_funding": "${:,.2f}",
        "funding_share_percent": "{:.1f}%"
    })
)

,project_groupings,total_funding,project_count,average_funding,funding_share_percent
3,Not Specified,"$122,974,253.82",64,"$1,921,472.72",59.2%
0,Boko Haram crisis 2014-,"$68,406,376.00",58,"$1,179,420.28",32.9%
1,Covid-19 2020-2021,"$15,000,005.00",1,"$15,000,005.00",7.2%
2,Ebola in western Africa 2014-2015,"$1,458,309.00",2,"$729,154.50",0.7%


In [29]:
top_projects = (
    df_clean.nlargest(
        10,
        "total_amount_approved"
    )[
        [
            "project_code",
            "project_title",
            "agency_name",
            "year",
            "emergency_type_name",
            "window_full_name",
            "total_amount_approved"
        ]
    ]
    .reset_index(drop=True)
)

display(
    top_projects.style.format({
        "total_amount_approved": "${:,.2f}"
    })
)

,project_code,project_title,agency_name,year,emergency_type_name,window_full_name,total_amount_approved
0,20-RR-WFP-056,Food Assistance to the most vulnerable people affected by COVID-19 in the northeast Nigeria,World Food Programme,2021,Displacement,Rapid Response,"$15,000,005.00"
1,22-RR-WFP-031,Food assistance to the most vulnerable conflict affected people in the northeast Nigeria,World Food Programme,2022,Violence/Clashes,Rapid Response,"$10,000,000.00"
2,22-UF-CEF-064,Multisectoral Response to the communities affected by malnutrition,United Nations Children’s Fund,2022,Violence/Clashes,Underfunded Emergencies,"$8,500,000.00"
3,23-RR-WFP-022,Provision of life-saving nutrition sensitive assistance to crisis affected population in Local Government Areas with famine-like situations in NE Nigeria.,World Food Programme,2023,Violence/Clashes,Rapid Response,"$6,000,000.00"
4,16-RR-WFP-041,"Life Saving Food Assistance to Extremely Vulnerable Nigerian IDPs in Borno and Yobe States, including Blanket Supplementary Feeding for Children at Risk aged 6-23 months",World Food Programme,2016,Displacement,Rapid Response,"$5,995,380.00"
5,CERF-NGA-24-RR-WFP-32768,Provision of life-saving nutrition-sensitive assistance to crisis affected population in Local Government Areas in NE Nigeria,World Food Programme,2024,Drought,Rapid Response,"$5,500,000.35"
6,22-RR-CEF-035,Multisectoral Rapid Response to the people affected by the lean season,United Nations Children’s Fund,2022,Violence/Clashes,Rapid Response,"$5,000,000.00"
7,20-UF-CEF-056,Child Protection and Education in Emergency response for children affected by conflict in Northeast of Nigeria,United Nations Children’s Fund,2020,Displacement,Underfunded Emergencies,"$4,399,657.00"
8,17-UF-WFP-010,Emergency Operation to provide Lifesaving support to households directly affected by insecurity in NE Nigeria,World Food Programme,2017,Displacement,Underfunded Emergencies,"$4,324,678.00"
9,21-RR-CEF-047,Cholera Prevention and Control through community health and WASH interventions in Northern Nigeria,United Nations Children’s Fund,2021,Cholera,Rapid Response,"$4,300,000.00"


In [30]:
sector_summary.to_csv(
    project_root
    / "outputs"
    / "tables"
    / "sector_combination_summary.csv",
    index=False
)

crisis_summary.to_csv(
    project_root
    / "outputs"
    / "tables"
    / "crisis_grouping_summary.csv",
    index=False
)

top_projects.to_csv(
    project_root
    / "outputs"
    / "tables"
    / "top_ten_projects.csv",
    index=False
)

print("Sector, crisis and top-project tables saved.")

Sector, crisis and top-project tables saved.


In [31]:
sector_chart_data = (
    sector_summary.head(10)
    .sort_values(
        "total_funding",
        ascending=True
    )
    .copy()
)

sector_chart_data["sector_label"] = (
    sector_chart_data["project_sectors"]
    .map(
        lambda text:
        text if len(text) <= 42
        else text[:39] + "..."
    )
)

sector_chart_data["funding_label"] = (
    sector_chart_data["total_funding"]
    .map(lambda value: f"${value / 1_000_000:.1f}M")
)

fig_sector = px.bar(
    sector_chart_data,
    x="total_funding",
    y="sector_label",
    orientation="h",
    text="funding_label",
    title="Top Sector Combinations by Total CERF Funding",
    labels={
        "total_funding": "Total approved funding (US$)",
        "sector_label": ""
    },
    hover_data={
        "project_sectors": True,
        "project_count": True,
        "average_funding": ":$,.2f",
        "funding_share_percent": ":.1f",
        "funding_label": False
    },
    color_discrete_sequence=["#287D8E"]
)

fig_sector.update_traces(
    textposition="outside",
    cliponaxis=False
)

fig_sector.update_layout(
    height=620,
    showlegend=False,
    margin=dict(l=80, r=80, t=90, b=70),
    xaxis=dict(
        tickprefix="$",
        tickformat=".2s"
    ),
    yaxis=dict(
        title="",
        automargin=True
    )
)

fig_sector.update_xaxes(
    range=[
        0,
        sector_chart_data["total_funding"].max() * 1.20
    ]
)

fig_sector.show()

In [32]:
crisis_chart_data = crisis_summary.sort_values(
    "total_funding",
    ascending=True
).copy()

crisis_chart_data["funding_label"] = (
    crisis_chart_data["total_funding"]
    .map(lambda value: f"${value / 1_000_000:.1f}M")
)

crisis_chart_data["grouping_status"] = np.where(
    crisis_chart_data["project_groupings"]
    == "Not Specified",
    "Not Specified",
    "Named crisis"
)

fig_crisis = px.bar(
    crisis_chart_data,
    x="total_funding",
    y="project_groupings",
    orientation="h",
    text="funding_label",
    color="grouping_status",
    title="CERF Funding by Recorded Crisis Grouping",
    labels={
        "total_funding": "Total approved funding (US$)",
        "project_groupings": "",
        "grouping_status": "Grouping status"
    },
    color_discrete_map={
        "Not Specified": "#E9A23B",
        "Named crisis": "#185A8D"
    },
    hover_data={
        "project_count": True,
        "average_funding": ":$,.2f",
        "funding_share_percent": ":.1f",
        "funding_label": False
    }
)

fig_crisis.update_traces(
    textposition="outside",
    cliponaxis=False
)

fig_crisis.update_layout(
    height=500,
    margin=dict(l=80, r=80, t=90, b=70),
    xaxis=dict(
        tickprefix="$",
        tickformat=".2s"
    ),
    yaxis=dict(
        title="",
        automargin=True
    )
)

fig_crisis.update_xaxes(
    range=[
        0,
        crisis_chart_data["total_funding"].max() * 1.20
    ]
)


In [33]:
fig_crisis.update_layout(
    height=500,
    showlegend=False,
    title=dict(
        text="CERF Funding by Recorded Crisis Grouping",
        x=0.5,
        xanchor="center"
    ),
    margin=dict(
        l=100,
        r=100,
        t=90,
        b=70
    ),
    yaxis=dict(
        title="",
        automargin=True
    )
)

fig_crisis.show()


In [34]:
top_projects_chart = (
    top_projects
    .sort_values(
        "total_amount_approved",
        ascending=True
    )
    .copy()
)

top_projects_chart["funding_label"] = (
    top_projects_chart["total_amount_approved"]
    .map(lambda value: f"${value / 1_000_000:.1f}M")
)

fig_projects = px.bar(
    top_projects_chart,
    x="total_amount_approved",
    y="project_code",
    orientation="h",
    text="funding_label",
    title="Ten Largest CERF-Funded Projects",
    labels={
        "total_amount_approved": "Approved funding (US$)",
        "project_code": "Project code"
    },
    hover_data={
        "project_title": True,
        "agency_name": True,
        "year": True,
        "emergency_type_name": True,
        "window_full_name": True,
        "funding_label": False
    },
    color_discrete_sequence=["#185A8D"]
)

fig_projects.update_traces(
    textposition="outside",
    cliponaxis=False
)

fig_projects.update_layout(
    height=620,
    showlegend=False,
    margin=dict(
        l=130,
        r=80,
        t=90,
        b=70
    ),
    xaxis=dict(
        tickprefix="$",
        tickformat=".2s"
    ),
    yaxis=dict(
        title="Project code",
        automargin=True
    )
)

fig_projects.update_xaxes(
    range=[
        0,
        top_projects_chart[
            "total_amount_approved"
        ].max() * 1.20
    ]
)

fig_projects.show()

In [35]:
# Identify the leading values for the executive summary
top_agency_funding = agency_summary.iloc[0]

most_projects_agency = (
    agency_summary
    .sort_values(
        "project_count",
        ascending=False
    )
    .iloc[0]
)

highest_average_agency = (
    agency_summary
    .sort_values(
        "average_funding",
        ascending=False
    )
    .iloc[0]
)

top_emergency = emergency_summary.iloc[0]
top_sector = sector_summary.iloc[0]

peak_year = (
    year_summary
    .sort_values(
        "total_funding",
        ascending=False
    )
    .iloc[0]
)

largest_project = top_projects.iloc[0]

print("EXECUTIVE FINDINGS")
print(
    "Total approved funding:",
    f"${clean_funding_total:,.2f}"
)
print(
    "Leading agency by funding:",
    top_agency_funding["agency_name"],
    f"(${top_agency_funding['total_funding']:,.2f})"
)
print(
    "Agency with most projects:",
    most_projects_agency["agency_name"],
    f"({most_projects_agency['project_count']} projects)"
)
print(
    "Highest average funding per project:",
    highest_average_agency["agency_name"],
    f"(${highest_average_agency['average_funding']:,.2f})"
)
print(
    "Leading emergency by funding:",
    top_emergency["emergency_type_name"],
    f"(${top_emergency['total_funding']:,.2f})"
)
print(
    "Leading sector combination:",
    top_sector["project_sectors"],
    f"(${top_sector['total_funding']:,.2f})"
)
print(
    "Peak funding year:",
    int(peak_year["year"]),
    f"(${peak_year['total_funding']:,.2f})"
)
print(
    "Largest project:",
    largest_project["project_code"],
    f"(${largest_project['total_amount_approved']:,.2f})"
)
print(
    "Top-two agency funding concentration:",
    f"{top_two_agency_share:.1f}%"
)

EXECUTIVE FINDINGS
Total approved funding: $207,838,943.82
Leading agency by funding: United Nations Children’s Fund ($73,198,383.67)
Agency with most projects: United Nations Children’s Fund (39 projects)
Highest average funding per project: World Food Programme ($4,052,510.39)
Leading emergency by funding: Displacement ($109,403,629.00)
Leading sector combination: Food Assistance ($46,820,063.35)
Peak funding year: 2021 ($33,500,110.00)
Largest project: 20-RR-WFP-056 ($15,000,005.00)
Top-two agency funding concentration: 64.5%


In [36]:
funding_category_summary = (
    df_clean
    .groupby(
        "funding_category",
        observed=True,
        as_index=False
    )
    .agg(
        project_count=(
            "project_id",
            "count"
        ),
        total_funding=(
            "total_amount_approved",
            "sum"
        )
    )
)

funding_category_summary["funding_share_percent"] = (
    funding_category_summary["total_funding"]
    / clean_funding_total
    * 100
)

display(
    funding_category_summary.style.format({
        "total_funding": "${:,.2f}",
        "funding_share_percent": "{:.1f}%"
    })
)

funding_category_summary.to_csv(
    project_root
    / "outputs"
    / "tables"
    / "funding_category_summary.csv",
    index=False
)

,funding_category,project_count,total_funding,funding_share_percent
0,Small (up to $500K),31,"$10,209,030.04",4.9%
1,Medium ($500K to $1M),29,"$21,762,279.37",10.5%
2,Large ($1M to $3M),46,"$76,255,226.25",36.7%
3,Very Large (over $3M),19,"$99,612,408.16",47.9%


In [37]:
data_quality_summary = pd.DataFrame({
    "field": [
        "Project grouping",
        "Project CAP code"
    ],
    "missing_count": [
        df_raw["projectgroupings"].isna().sum(),
        df_raw["projectcapcodes"].isna().sum()
    ]
})

data_quality_summary["missing_percentage"] = (
    data_quality_summary["missing_count"]
    / len(df_raw)
    * 100
)

data_quality_summary["percentage_label"] = (
    data_quality_summary["missing_percentage"]
    .map(lambda value: f"{value:.1f}%")
)

fig_quality = px.bar(
    data_quality_summary,
    x="missing_percentage",
    y="field",
    orientation="h",
    text="percentage_label",
    title="Missingness in Key Raw-Data Fields",
    labels={
        "missing_percentage": "Missing records (%)",
        "field": ""
    },
    color_discrete_sequence=["#E9A23B"]
)

fig_quality.update_traces(
    textposition="outside"
)


fig_quality.update_layout(
    height=380,
    showlegend=False,
    margin=dict(l=80, r=70, t=90, b=60),
    xaxis=dict(
        range=[0, 100],
        ticksuffix="%"
    ),
    yaxis=dict(
        title="",
        automargin=True
    )
)

fig_quality.show()